# Titanic Survival Data Analysis

This project analyzes Titanic passenger data using Python. The goal is to understand which factors were related to passenger survival and to build a simple predictive model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Load and Inspect the Data


In [ ]:
df = pd.read_csv('../data/titanic.csv')
df.head()


In [ ]:
print(df.shape)
df.info()


In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


## Data Cleaning


In [ ]:
titanic = df.copy()
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
if 'Embarked' in titanic.columns:
    titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])
if 'Cabin' in titanic.columns:
    titanic = titanic.drop(columns=['Cabin'])
titanic = titanic.dropna()
titanic.isnull().sum()


In [ ]:
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['IsAlone'] = np.where(titanic['FamilySize'] == 1, 1, 0)
titanic[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()


## Research Question 1: What was the overall survival rate?


In [ ]:
overall_survival_rate = titanic['Survived'].mean()
print('Overall Survival Rate:', round(overall_survival_rate * 100, 2), '%')


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=titanic, x='Survived')
plt.title('Passenger Survival Count')
plt.xlabel('Survived')
plt.ylabel('Number of Passengers')
plt.xticks([0, 1], ['Did Not Survive', 'Survived'])
plt.show()


## Research Question 2: Did passenger sex affect survival rate?


In [ ]:
survival_by_sex = titanic.groupby('Sex')['Survived'].mean()
survival_by_sex


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=titanic, x='Sex', y='Survived')
plt.title('Survival Rate by Sex')
plt.xlabel('Sex')
plt.ylabel('Survival Rate')
plt.show()


## Research Question 3: Did passenger class affect survival rate?


In [ ]:
survival_by_class = titanic.groupby('Pclass')['Survived'].mean()
survival_by_class


In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=titanic, x='Pclass', y='Survived')
plt.title('Survival Rate by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Survival Rate')
plt.show()


## Research Question 4: How were age, fare, and family size related to survival?


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=titanic, x='Age', bins=30, kde=True)
plt.title('Age Distribution of Titanic Passengers')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=titanic, x='Age', hue='Survived', bins=30, kde=True)
plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=titanic, x='Survived', y='Fare')
plt.title('Fare Distribution by Survival')
plt.xlabel('Survived')
plt.ylabel('Fare')
plt.xticks([0, 1], ['Did Not Survive', 'Survived'])
plt.show()


In [ ]:
family_survival = titanic.groupby('FamilySize')['Survived'].mean()
family_survival


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=titanic, x='FamilySize', y='Survived')
plt.title('Survival Rate by Family Size')
plt.xlabel('Family Size')
plt.ylabel('Survival Rate')
plt.show()


## Correlation Heatmap


In [ ]:
numeric_columns = titanic.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_columns.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## Statistical Test


In [ ]:
contingency_table = pd.crosstab(titanic['Sex'], titanic['Survived'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print('Chi-square statistic:', chi2)
print('P-value:', p_value)
if p_value < 0.05:
    print('There is a statistically significant relationship between sex and survival.')
else:
    print('There is not a statistically significant relationship between sex and survival.')


## Research Question 5: Can a simple machine learning model predict survival?


In [ ]:
model_data = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone']].copy()
model_data['Sex'] = model_data['Sex'].map({'male': 0, 'female': 1})
X = model_data[['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone']]
y = model_data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print('Model Accuracy:', round(accuracy * 100, 2), '%')


In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
feature_importance = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_[0]})
feature_importance = feature_importance.sort_values(by='Coefficient', ascending=False)
feature_importance


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=feature_importance, x='Coefficient', y='Feature')
plt.title('Logistic Regression Feature Coefficients')
plt.xlabel('Coefficient')
plt.ylabel('Feature')
plt.show()


# Conclusion

This project analyzed Titanic passenger survival using Python. The analysis showed that passenger sex and passenger class were strongly related to survival outcomes. Female passengers had a higher survival rate than male passengers, and first-class passengers had a higher survival rate than passengers in lower classes.

Age, fare, and family size also provided useful information. Passengers who paid higher fares generally had better survival outcomes, which may be related to passenger class and access to lifeboats.

The machine learning model used logistic regression to predict survival. The model used passenger class, sex, age, fare, family size, and whether the passenger was alone as features. The model achieved a reasonable accuracy score, showing that these variables can help predict survival.

Overall, this project demonstrates how Python can be used to import, clean, analyze, visualize, and model data in order to answer analytical questions.
